In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


In [3]:
df = pd.read_parquet("d:/FIAP/Engenharia de Machine Learning/Fase 3/tech_challenge_3/data/processed/flights_processed.parquet")

In [5]:
pd.set_option('display.max_columns', None) 
df.head()

,AIRLINE,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE_CODE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,AIRPORT,CITY,STATE,LATITUDE,LONGITUDE,DATE,HOUR,IS_DELAYED,IS_SHORT_DISTANCE,IS_MEDIUM_DISTANCE,IS_LONG_DISTANCE,IS_MORNING,IS_AFTERNOON,IS_NIGHT,IS_HOLIDAY,IS_WINTER,IS_SPRING,IS_SUMMER,IS_FALL
0,AS,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,-11.0,205.0,1448,430,Ted Stevens Anchorage International Airport,Anchorage,AK,61.17432,-149.99619,2015-01-01,0,0,0,1,0,1,0,0,1,1,0,0,0
1,AA,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,-8.0,280.0,2330,750,Los Angeles International Airport,Los Angeles,CA,33.94254,-118.40807,2015-01-01,0,0,0,0,1,1,0,0,1,1,0,0,0
2,US,2015,1,1,4,US,840,N171US,SFO,CLT,20,-2.0,286.0,2296,806,San Francisco International Airport,San Francisco,CA,37.61900,-122.37484,2015-01-01,0,0,0,0,1,1,0,0,1,1,0,0,0
3,AA,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,-5.0,285.0,2342,805,Los Angeles International Airport,Los Angeles,CA,33.94254,-118.40807,2015-01-01,0,0,0,0,1,1,0,0,1,1,0,0,0
4,AS,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,-1.0,235.0,1448,320,Seattle-Tacoma International Airport,Seattle,WA,47.44898,-122.30931,2015-01-01,0,0,0,1,0,1,0,0,1,1,0,0,0


In [21]:
# Filtrar voos atrasados
delayed_flights_origin = df[df["IS_DELAYED"] == 1]

# Contar atrasos por aeroporto de origem
delays_by_airport_origin = delayed_flights_origin["ORIGIN_AIRPORT"].value_counts()

# Proporção em relação ao total de atrasos
delays_by_airport_origin_percent = delays_by_airport_origin / delays_by_airport_origin.sum() * 100

top10_airports = delays_by_airport_origin_percent.head(10).index  # Top 10 aeroportos

# Criar coluna nova
df["ORIGIN_AIRPORT_TOP10"] = df["ORIGIN_AIRPORT"].where(df["ORIGIN_AIRPORT"].isin(top10_airports), "Other")

In [30]:
# Filtrar voos atrasados
delayed_flights_destination = df[df["IS_DELAYED"] == 1]

# Contar atrasos por aeroporto de origem
delays_by_airport_destination = delayed_flights_destination["DESTINATION_AIRPORT"].value_counts()

# Proporção em relação ao total de atrasos
delays_by_airport_percent = delays_by_airport_destination / delays_by_airport_destination.sum() * 100

top10_airports_dest = delays_by_airport_percent.head(10).index  # Top 10 aeroportos

# Converter para string
df["DESTINATION_AIRPORT"] = df["DESTINATION_AIRPORT"].astype(str)

# Criar coluna Top10 + Other
df["DESTINATION_AIRPORT_TOP10"] = df["DESTINATION_AIRPORT"].where(df["DESTINATION_AIRPORT"].isin(top10_airports_dest), "Other")

In [38]:
# Filtrar voos atrasados
delayed_flights = df[df["IS_DELAYED"] == 1]

# Contar atrasos por aeroporto de origem
delays_by_airport = delayed_flights["CITY"].value_counts()

# Proporção em relação ao total de atrasos
delays_by_airport_percent = delays_by_airport / delays_by_airport.sum() * 100

top10_airports = delays_by_airport_percent.head(10).index  # Top 10 aeroportos

# Converter para string
df["CITY"] = df["CITY"].astype(str)

# Criar coluna Top10 + Other
df["CITY_TOP10"] = df["CITY"].where(df["CITY"].isin(top10_airports), "Other")

In [39]:
# Filtrar voos atrasados
delayed_flights = df[df["IS_DELAYED"] == 1]

# Contar atrasos por aeroporto de origem
delays_by_airport = delayed_flights["STATE"].value_counts()

# Proporção em relação ao total de atrasos
delays_by_airport_percent = delays_by_airport / delays_by_airport.sum() * 100

top10_airports = delays_by_airport_percent.head(10).index  # Top 10 aeroportos

# Converter para string
df["STATE"] = df["STATE"].astype(str)

# Criar coluna Top10 + Other
df["STATE_TOP10"] = df["STATE"].where(df["STATE"].isin(top10_airports), "Other")

In [44]:
# Filtrar voos atrasados
delayed_flights = df[df["IS_DELAYED"] == 1]

# Contar atrasos por aeroporto de origem
delays_by_airport = delayed_flights["TAIL_NUMBER"].value_counts()

# Proporção em relação ao total de atrasos
delays_by_airport_percent = delays_by_airport / delays_by_airport.sum() * 100

top10_airports = delays_by_airport_percent.head(10).index  # Top 10 aeroportos

# Converter para string
df["TAIL_NUMBER"] = df["TAIL_NUMBER"].astype(str)

# Criar coluna Top10 + Other
df["TAIL_NUMBER_TOP10"] = df["TAIL_NUMBER"].where(df["TAIL_NUMBER"].isin(top10_airports), "Other")

In [45]:
#separa features e target 
X = df.drop(columns=["IS_DELAYED", "DEPARTURE_DELAY", "DATE", "YEAR", "DAY", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "AIRLINE", "AIRPORT", "CITY", "STATE", "TAIL_NUMBER"])  # remove target e colunas que “vazam” info
y = df["IS_DELAYED"]

In [46]:
X.head()

,MONTH,DAY_OF_WEEK,AIRLINE_CODE,FLIGHT_NUMBER,SCHEDULED_DEPARTURE,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,LATITUDE,LONGITUDE,HOUR,IS_SHORT_DISTANCE,IS_MEDIUM_DISTANCE,IS_LONG_DISTANCE,IS_MORNING,IS_AFTERNOON,IS_NIGHT,IS_HOLIDAY,IS_WINTER,IS_SPRING,IS_SUMMER,IS_FALL,ORIGIN_AIRPORT_TOP10,DESTINATION_AIRPORT_TOP10,CITY_TOP10,STATE_TOP10,TAIL_NUMBER_TOP10
0,1,4,AS,98,5,205.0,1448,430,61.17432,-149.99619,0,0,1,0,1,0,0,1,1,0,0,0,Other,Other,Other,Other,Other
1,1,4,AA,2336,10,280.0,2330,750,33.94254,-118.40807,0,0,0,1,1,0,0,1,1,0,0,0,LAX,Other,Los Angeles,CA,Other
2,1,4,US,840,20,286.0,2296,806,37.61900,-122.37484,0,0,0,1,1,0,0,1,1,0,0,0,SFO,Other,San Francisco,CA,Other
3,1,4,AA,258,20,285.0,2342,805,33.94254,-118.40807,0,0,0,1,1,0,0,1,1,0,0,0,LAX,Other,Los Angeles,CA,Other
4,1,4,AS,135,25,235.0,1448,320,47.44898,-122.30931,0,0,1,0,1,0,0,1,1,0,0,0,Other,Other,Other,Other,Other


In [47]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [48]:
for col in ["AIRLINE_CODE", "ORIGIN_AIRPORT_TOP10", "DESTINATION_AIRPORT_TOP10", "CITY_TOP10", "STATE_TOP10", "TAIL_NUMBER_TOP10"]:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

In [49]:
# Criar o modelo
model = RandomForestClassifier(
    n_estimators=100,        # número de árvores
    random_state=42,
    class_weight="balanced"  # importante em targets desbalanceados
)

# Treinar
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]  # probabilidade de atraso 

In [ ]:
# Relatório completo
print(classification_report(y_test, y_pred))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
print(cm)

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC:", roc_auc)

In [ ]:
feat_imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(feat_imp.head(20))